# Lab 5: Localization

A robot is placed somewhere in a maze and is not told where it is. It can
drive in four directions, and it can check for walls on its four sides with
a sonar that reports the wrong answer about one time in ten.

In this lab you will:

1. Write the sonar model, which gives the probability of a reading in a given cell.
2. Implement the **predict** step, which is the marginalization from Lab 4 applied over time.
3. Implement the **update** step, which is the conditioning from Lab 4 applied over time.
4. Combine the two into a loop, and track a robot that starts with no information about its position.
5. Give the filter the wrong sonar model, and measure what that costs.

**Time:** about 2.5 hours.

**Reading:** AIMA sections 14.1 and 14.2.

**What to submit:** commit this notebook with its outputs, plus `results_trace.csv`, `results_runs.csv` and `predictions.json`.

<div style="border-left: 4px solid #0072B2; padding: 0.4em 1em; background: #f4f8fb;">

### How to use this notebook

* **Stub cells** hold a signature, a docstring, and `raise NotImplementedError`. Delete that line and write your code.
* **Check cells** run right after each stub. They name the position that broke, what should have come out, and what to look at. Add `verbose=True` for a full traceback.
* **Hints** are in collapsible blocks, ordered from a nudge to near-pseudocode.
* **Provided cells** are marked. You never write plotting, tournament, or export code.

</div>

## Part 0: The robot's situation


Run the next cell first. It checks that this notebook is running in the course environment with everything the lab needs.

In [ ]:
%run check_environment.py

In [ ]:
# PROVIDED
import numpy as np
import pandas as pd

from lab5kit.world import DIRECTIONS, Maze
from lab5kit.measures import cells_covering, entropy, top_cells, total
from lab5kit import experiments as experiment
from lab5kit import viz
from lab5kit.predictions import record_predictions, reveal
from lab5kit.results import save_results
from lab5kit.checks import (
    check_filter_sequence,
    check_predict,
    check_sensor_likelihood,
    check_update,
)

world = Maze()

beliefs = trace = many = runs = None


def ready(*results):
    for result in results:
        if result is None:
            print("⏭  Skipped for now. Run the cell above first, then "
                  "come back to this one.")
            return False
    return True


print(world)
print(world.render())

The robot is in one of 30 free cells. 

Everything the robot does is described by one of two models. The robot has a motion model and a sensor model:

| call | what it is | what it tells you |
|---|---|---|
| `world.transition(cell, action)` | the **motion model** | where a move from `cell` might land |
| `world.signature(cell)` | the **sensor model**, noise-free | what a perfect sonar would read in `cell` |
| `world.epsilon` | the sonar's reliability | how often one beam is wrong |

A **cell** is a `(row, col)` pair, with row 0 at the top. Driving is
unreliable. The robot moves in the direction it was given 80% of the time,
and drifts to one of the two sides otherwise. Any move into a wall leaves
the robot where it started.

In [ ]:
# PROVIDED
viz.show_maze(world, path=None, title="the maze, with the walls in grey")

print("one step east from (1, 1):")
for landing, probability in sorted(world.transition((1, 1), "east").items()):
    print(f"   {landing}  {probability:.2f}")

print("\none step west from (1, 1), which has a wall to the west:")
for landing, probability in sorted(world.transition((1, 1), "west").items()):
    print(f"   {landing}  {probability:.2f}")

### What the sonar says

The sonar has four beams, one per side, in the order of `DIRECTIONS`:
north, east, south, west. Each beam answers a single question, "is there a
wall on this side?", and reports `1` for yes.

A **reading** is therefore four bits, such as `(1, 0, 1, 0)`. That reading
says there are walls to the north and south and open space to the east and
west, which describes a corridor running from left to right.

Each beam is wrong independently with probability `world.epsilon`.

In [ ]:
# PROVIDED
print(f"DIRECTIONS = {DIRECTIONS}")
print(f"epsilon    = {world.epsilon}\n")
for cell in [(1, 1), (1, 2), (3, 4), (5, 8)]:
    print(f"a perfect sonar in {cell} reads {world.signature(cell)}")

rng = np.random.default_rng(0)
honest = world.signature((1, 2))
print(f"\nten real readings from (1, 2), whose true signature is {honest}:")
lies = 0
for _ in range(10):
    heard = world.sample_reading((1, 2), rng)
    wrong = sum(1 for a, b in zip(heard, honest) if a != b)
    lies += wrong
    print("   ", heard, "  " + "x" * wrong)
print(f"\n{lies} of those 40 beams lied.")

Five of those forty beams were wrong, which is close to what a 10% error
rate should produce. Notice what that does to the readings themselves. Four
of the ten readings are wrong somewhere, so most of the time the robot is
being told something false about one of its sides.

### One reading is not a location

Even a perfect sonar would not be enough. Four bits can say only so much
about thirty cells, and the maze contains many places that feel identical.

In [ ]:
# PROVIDED
viz.show_ambiguity(world);

reading = (1, 0, 1, 0)
twins = world.matching_cells(reading)
print(f"{len(twins)} cells would give a perfect sonar the reading {reading}:")
print("   ", twins)

Those eight cells are indistinguishable from one another while the robot
stands still. Its only way out of this is to move. Motion is what breaks
the ambiguity, because two cells that read the same usually have different
cells around them.

### A belief

The robot cannot know where it is, so it keeps a probability distribution
over all the places it could be. We will call this a **belief**: a
dictionary with one entry per free cell, summing to 1.

Knowing nothing corresponds to a uniform belief, which is 1/30 in every
cell.

In [ ]:
# PROVIDED
belief = world.uniform_belief()
print(f"{len(belief)} cells, summing to {total(belief):.6f}")
print(f"entropy {entropy(belief):.2f} bits (this is a measure of our ignorance), "
      f"{cells_covering(belief, 0.9)} cells needed to cover 90%")
viz.show_belief(world, belief, title="knowing nothing: 1/30 everywhere");

That picture is the only thing that changes in this lab. Every step from
here on is one of two operations:

* the robot **drives**, and the belief spreads out. This is marginalizing over where the robot used to be.
* the robot **reads its sonar**, and the belief sharpens. This is conditioning on what it saw.

You wrote both of these in Lab 4, using a table of Titanic passengers. The
only new element is that there is now a before and an after.

## Part 1: Predictions

*About 5 minutes.*

Record three guesses before you build anything. 

1. **`readings_to_localize`**. Starting from the uniform belief above, how many sonar readings does the filter need before it places at least half its probability on the cell the robot is really in? Give a whole number.
2. **`best_assumed_epsilon`**. In Part 6 the sonar is replaced by a worse one that is wrong 30% of the time. You will run filters that have been told the error rate is 1%, 5%, 10%, 20%, 30%, 40% and 50%. Which of those filters will guess correctly most often?
3. **`overconfident_is_surer`**. One of those filters has been told that its sonar is almost never wrong, when in fact it is wrong 30% of the time. Will that filter end up more sure of its own best guess than the filter that was told the truth? Answer `"yes"`, `"no"` or `"same"`.

In [ ]:
predictions = record_predictions(
    readings_to_localize=None,       # a whole number, 1 or more
    best_assumed_epsilon=None,       # one of 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5
    overconfident_is_surer=None,     # "yes", "no" or "same"
    why="",                          # a sentence or two
)

## Part 2: The sonar model

Before the robot can learn anything from a reading, it must be able to say
how likely that reading would be in a given cell. That quantity is
$P(z \mid s)$, the probability of reading $z$ if the robot is in cell $s$.

The four beams fail independently, so the four factors multiply:

$$P(z \mid s) = \prod_{d \in \text{directions}} \begin{cases} 1 - \epsilon & \text{if beam } d \text{ agrees with } s \\ \epsilon & \text{if it disagrees} \end{cases}$$

This product is the same assumption you made in Lab 4, where you multiplied
one probability per word in a text message. There the assumption was naive,
because words are not independent. Here it is correct, because the four
beams really do fail separately.

Note two things that this function is **not**:

* It is not a probability over cells. The values of $P(z \mid s)$ taken over all cells $s$ do not sum to 1, and they should not.
* It is not 0 for a surprising reading. Every reading is possible in every cell, although not equally likely.

### Task 1

Compare the reading against `world.signature(cell)` beam by beam.

In [ ]:
def sensor_likelihood(world, reading, cell):
    """The probability of a sonar reading, given where the robot is.

    Args:
        world: The Maze. Its ``epsilon`` is how often one beam lies, and
            ``world.signature(cell)`` is what a perfect sonar would read.
        reading: A 4-tuple of 0s and 1s, in the order of ``DIRECTIONS``.
        cell: The cell to evaluate the reading against.

    Returns:
        P(reading | cell), a single number between 0 and 1.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_sensor_likelihood(sensor_likelihood)

<details><summary>Hint 1: the shape of it</summary>

Start a running product at `1.0`, walk through the four beams, and multiply
in one factor for each. `zip(reading, world.signature(cell))` gives you the
pairs to compare.

</details>

<details><summary>Hint 2: which factor</summary>

For each pair there are only two cases. If the beam agrees with the true
signature, it reported correctly, which happens with probability
`1 - world.epsilon`. If it disagrees, it reported incorrectly, which happens
with probability `world.epsilon`.

</details>

<details><summary>Hint 3: near-pseudocode</summary>

```
truth = world.signature(cell)
likelihood = 1.0
for observed, actual in zip(reading, truth):
    if observed == actual:
        multiply likelihood by (1 - world.epsilon)
    else:
        multiply likelihood by world.epsilon
return likelihood
```

</details>

The cell below evaluates the same reading in every cell at once. This is the
evidence the robot has to work with, laid out over the maze.

In [ ]:
# PROVIDED
likelihoods = {cell: sensor_likelihood(world, reading, cell)
               for cell in world.cells}
print(f"reading {reading}; these numbers add up to {sum(likelihoods.values()):.3f}, "
      "which is not 1 and is not meant to be")
viz.show_belief(world, likelihoods,
                title=f"P({reading} | cell), for every cell");

The eight corridor cells are the darkest, at $0.9^4 = 0.656$. Their
neighbours are a factor of nine dimmer for each beam they disagree with. No
cell is completely black. Even a cell that disagrees on all four beams has
probability $0.1^4$, or one chance in ten thousand, of producing this
reading.

Note that readings are evidence and not proof. The robot should be able to recover even from a bad reading.

## Part 3: Driving, the predict step

*About 25 minutes.*

The robot drives one step. It has taken no new reading, so it has learned
nothing, but it has moved, so the set of places it might be has changed.

If the robot was in cell $s$ and drove $a$, the probability that it is now
in $s'$ is `world.transition(s, a)[s']`. The robot does not know which $s$
it was in, so it must consider all of them, weighted by how likely each one
was:

$$P(s') = \sum_{s} P(s' \mid s, a) \, P(s)$$

That sum is a marginalization. In Lab 4 you dropped `Pclass` by adding up
over its values. Here you drop the robot's previous position by adding up
over the cells it might have been in.

Stated mechanically: every cell hands out all of its probability to the
cells it could move to, divided in the proportions the motion model gives.
Nothing is created and nothing is lost, so the belief still sums to 1.

### Task 2

Return a **new** belief. Do not modify the one you were given, because later
parts of the notebook reuse it.

In [ ]:
def predict(world, belief, action):
    """The belief after driving one step, before any new reading.

    Args:
        world: The Maze, whose ``transition`` is the motion model.
        belief: A cell -> probability dictionary, summing to 1.
        action: One of ``DIRECTIONS``.

    Returns:
        A new cell -> probability dictionary over every cell in
        ``world.cells``, summing to 1.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_predict(predict)

<details><summary>Hint 1: start from the answer's shape</summary>

The result has an entry for every free cell, including the cells that end up
at 0. Build it empty first, with `{cell: 0.0 for cell in world.cells}`, and
add into it.

</details>

<details><summary>Hint 2: push, do not pull</summary>

It is tempting to loop over the *new* cells and ask "where could I have come
from?". Answering that needs the motion model backwards, which you do not
have.

Loop over the *old* cells instead, and push their probability forward. For
each cell the robot might have been in, `world.transition(cell, action)`
says where that probability goes and in what proportions.

</details>

<details><summary>Hint 3: near-pseudocode</summary>

```
moved = a dictionary with 0.0 for every cell in world.cells
for each cell, probability in belief:
    for each landing, chance in world.transition(cell, action):
        add probability * chance to moved[landing]
return moved
```

Note the `+=`. Several cells can send probability to the same landing cell,
and all of it counts.

</details>

### What driving blind does

Suppose the robot starts out certain that it is in `(1, 1)`, and then drives
east five times without ever taking a reading. At each step a little
probability drifts sideways or is stopped against a wall.

In [ ]:
# PROVIDED
blind = [world.point_belief((1, 1))]
for _ in range(5):
    blind.append(predict(world, blind[-1], "east"))

figure, axes = viz.plt.subplots(1, len(blind), figsize=(2.5 * len(blind), 2.9))
for step, (ax, state) in enumerate(zip(axes, blind)):
    viz.show_belief(world, state, ax=ax, vmax=1.0,
                    title=f"{step} moves east\n{cells_covering(state, 0.9)} cells cover 90%")
    ax.set_xticks([])
    ax.set_yticks([])
figure.suptitle("driving with your eyes shut", fontsize=10, color="0.3")
figure.tight_layout()

Certainty decays with every move. After five blind moves the robot has only
a vague idea that it is somewhere on the right-hand side of the maze.

This is called **dead reckoning**, and it happens to every robot, aircraft
and phone that tries to track itself from its own movements alone. Something
has to push in the other direction.

## Part 4: Looking, the update step

*About 25 minutes.*

The robot takes a reading. It now has evidence, and Bayes' rule says what to
do with it:

$$P(s \mid z) = \frac{P(z \mid s) \, P(s)}{P(z)}, \qquad P(z) = \sum_{s} P(z \mid s) \, P(s)$$

In words, and this is the whole of it: multiply each cell's probability by
how well that cell explains the reading, then divide everything by the total
so that it sums to 1 again.

The division is the part that goes wrong. $P(z)$ is not a quantity you look
up. It is the sum of the numbers you have just computed. Lab 4 made the same
point with Titanic passengers: a conditional probability is a share of what
is still in play, and dividing by the wrong total is the classic mistake.

### Task 3

Use your `sensor_likelihood` from Part 2.

In [ ]:
def update(world, belief, reading):
    """The belief after taking one sonar reading.

    Args:
        world: The Maze, whose ``epsilon`` says how noisy the sonar is.
        belief: A cell -> probability dictionary, summing to 1.
        reading: A 4-tuple of 0s and 1s, in the order of ``DIRECTIONS``.

    Returns:
        A new cell -> probability dictionary over every cell in
        ``world.cells``, summing to 1.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_update(update)

<details><summary>Hint 1: two passes</summary>

Do this in two steps, and do not try to merge them. First multiply every
cell's probability by its likelihood and keep the results. Then add those up
and divide each one by that sum.

You cannot normalize as you go, because you do not know the total until you
have been round once.

</details>

<details><summary>Hint 2: what to divide by</summary>

Divide by the total of the numbers you have just computed. Not by 30, and
not by the number of cells that match the reading, but by the sum of
`belief[cell] * sensor_likelihood(...)` over every cell. That total is
`P(reading)`, and dividing by it is the only thing that makes the result a
probability distribution again.

</details>

<details><summary>Hint 3: near-pseudocode</summary>

```
weighted = {}
for each cell, probability in belief:
    weighted[cell] = probability * sensor_likelihood(world, reading, cell)

evidence = sum of weighted's values
return {cell: value / evidence for each cell, value in weighted}
```

If that total comes out as exactly 0 there is nothing to divide by. This
cannot happen while `world.epsilon` is above 0. In Part 6 you will meet the
case where it can.

</details>

The cell below applies the same corridor reading to a robot that knew
nothing beforehand.

In [ ]:
# PROVIDED
after = update(world, world.uniform_belief(), reading)
print(f"before: entropy {entropy(world.uniform_belief()):.2f} bits, "
      f"{cells_covering(world.uniform_belief(), 0.9)} cells cover 90%")
print(f"after:  entropy {entropy(after):.2f} bits, "
      f"{cells_covering(after, 0.9)} cells cover 90%")
print(f"\nmost likely cells now:")
for cell, probability in top_cells(after, 4):
    print(f"   {cell}  {probability:.4f}")

figure, axes = viz.plt.subplots(1, 2, figsize=(9.5, 3.2))
viz.show_belief(world, world.uniform_belief(), ax=axes[0], vmax=0.12,
                title="before the reading")
viz.show_belief(world, after, ax=axes[1], vmax=0.12,
                title=f"after reading {reading}")
figure.tight_layout()

One reading has taken the robot from "anywhere" to "probably one of these
eight places". The eight corridor cells now hold most of the probability and
share it equally, which is what we should expect: nothing in a single
reading can tell them apart.

**Question.** The cells `(3, 4)` and `(3, 5)` sit in the open middle of the
maze with no wall on any side, so this reading, which reports walls to the
north and south, is wrong about them twice over. They still came out of the
update with about a thousandth of the belief each, rather than zero. Why?

*Your answer:*

## Part 5: The filter

We now combine the two steps into a loop. The robot reads, drives, reads and
drives again, and at every moment it holds a belief about where it is. This
is all that a **Bayes filter** is: predict, update, predict, update, for as
long as the robot is running.

The bookkeeping matters, so the timeline is written out below. Note that the
robot takes a reading before it moves at all.

```
t = 0:   reading[0]                                  -> belief[0]
t = 1:   action[0], then reading[1]                  -> belief[1]
t = 2:   action[1], then reading[2]                  -> belief[2]
  ...
```

There is always exactly one more reading than there are actions, and the run
**starts with an update, not a predict**, because nothing has moved yet.

### Task 4

Call your own `predict` and `update`. There is no new mathematics in this
task. The whole of it is getting the order and the indices right.

In [ ]:
def filter_sequence(world, belief, actions, readings):
    """Track the robot through a whole run, one belief per time step.

    Args:
        world: The Maze the filter believes in.
        belief: The belief to start from, before any reading.
        actions: The actions taken, in order. Length T.
        readings: The sonar readings, in order. Length T + 1. One is
            taken before anything moves, and one after each action.

    Returns:
        A list of T + 1 beliefs. Entry t is the belief after reading t has
        been taken into account, so entry 0 is ``belief`` updated on
        ``readings[0]`` alone.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_filter_sequence(filter_sequence)

<details><summary>Hint 1: before the loop</summary>

The first belief is not `belief` itself. It is `belief` after the robot has
looked around once. Do that update before the loop starts, and put the
result in your list.

</details>

<details><summary>Hint 2: inside the loop</summary>

Each turn of the loop consumes one action and the reading that follows it:
predict with the action, then update with the reading, then append the
result.

`zip(actions, readings[1:])` pairs them up correctly. If you prefer indices,
action `t - 1` goes with reading `t`.

</details>

<details><summary>Hint 3: near-pseudocode</summary>

```
current = update(world, belief, readings[0])
beliefs = [current]
for action, reading in zip(actions, readings[1:]):
    current = predict(world, current, action)
    current = update(world, current, reading)
    beliefs.append(current)
return beliefs
```

</details>

### Watching one robot find itself

The run below is fixed: thirteen readings and twelve moves, starting from no
information at all.

In [ ]:
# PROVIDED
trajectory = world.simulate(12, np.random.default_rng(7))
beliefs, trace = experiment.run_filter(world, trajectory, filter_sequence)
print(trace.to_string(index=False))

In [ ]:
# PROVIDED
if ready(beliefs):
    viz.show_filter_steps(world, beliefs, trajectory, times=[0, 2, 4, 6, 8, 12]);

The `p_truth` column tells the story of this run. The first reading leaves
eight candidates. By `t = 4` nearly half the belief is in the right place.
The filter then goes wrong. From `t = 5` to `t = 7` its best guess is one or
two cells off, and at `t = 6` it places under 11% on the truth. A reading
then arrives that only one cell can explain, and from `t = 8` onwards the
filter is correct and 94% sure.

**Question.** At `t = 6` the filter's best guess is `(5, 6)` when the robot
is really in `(5, 5)`, one cell away. Nothing was broken, and no reading was
misread by your code. Give one reason the filter can be wrong here, and say
whether more readings of the same kind would have corrected it.

*Your answer:*

### How long does it take, in general?

A single run proves nothing. The cell below runs thirty trajectories, each
with its own starting cell, its own random drives and its own sonar errors.
That is enough to answer prediction 1.

In [ ]:
# PROVIDED
many = experiment.run_many(world, filter_sequence, seeds=range(30), steps=15)
times = experiment.localization_times(many, threshold=0.5)
print(f"median readings until half the belief sits on the true cell: "
      f"{times.median():.0f}")
print(f"runs that never got there: {int(times.isna().sum())} of {len(times)}")
viz.plot_convergence(many, "p_truth");

In [ ]:
# PROVIDED
if ready(many):
    viz.plot_convergence(many, "cells90");

## Part 6: Giving the filter the wrong model

Everything so far has been honest. The filter's `world` was the same world
the readings came from: `world.epsilon` was 0.1, and the sonar really was
wrong one time in ten.

Real systems rarely work out that way. Somebody writes 0.1 into a
configuration file, and the sonar that is actually installed is dirtier than
the datasheet claimed. So we will keep the robot and the maze exactly as
they are, make the **real** sonar wrong 30% of the time, and see what
happens to a filter that has been told something else.

`world.variant(epsilon=...)` returns a copy of the maze with different
noise. The trajectory comes from the true world, and the filter is given the
variant.

In [ ]:
# PROVIDED
truth = world.variant(epsilon=0.3)
print(f"the world the data comes from: {truth}")
print(f"the world the filter believes: {world}")

### First, the extreme case

Tell the filter that its sonar is perfect by setting `epsilon=0.0`, and then
feed it readings from a sonar that is wrong three times in ten.

In [ ]:
# PROVIDED
certain = truth.variant(epsilon=0.0)
noisy_run = truth.simulate(12, np.random.default_rng(3))
try:
    filter_sequence(certain, certain.uniform_belief(),
                    noisy_run.actions, noisy_run.readings)
    print("no complaint, which means your update is not dividing by "
          "P(reading)")
except ValueError as error:
    print(f"ValueError: {error}")

A filter that believes its sonar cannot be wrong treats every reading as
proof. Cells are ruled out permanently, one at a time, until a reading
arrives that contradicts the last cell still standing. The belief is then
zero everywhere, there is nothing to divide by, and the robot's answer to
"where am I?" is nowhere at all.

The filter cannot recover from this. No evidence can bring a cell back from
zero, because zero times anything is zero. This is not a bug in the filter.
It is the filter doing exactly what it was told to do by a model that left
no room for being wrong.

### Now the whole range

The sweep below runs seven filters, told that the sonar is wrong 1%, 5%,
10%, 20%, 30%, 40% and 50% of the time. Every one of them sees **the same
thirty trajectories**, so no filter is helped by getting luckier data than
another.

This cell performs 210 filter runs and takes a few seconds.

In [ ]:
# PROVIDED
assumed = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
runs = experiment.run_mismatch_sweep(truth, filter_sequence, assumed,
                                     seeds=range(30), steps=30)
summary = runs.groupby("assumed_epsilon")[
    ["accuracy", "mean_confidence", "mean_p_truth", "mean_entropy"]].mean()
print(summary.round(3).to_string())
print(f"\nfor scale: guessing a cell at random would be right "
      f"{1 / len(world.cells):.1%} of the time")

In [ ]:
# PROVIDED
if ready(runs):
    viz.plot_mismatch(runs);

In [ ]:
# PROVIDED
if ready(runs):
    viz.plot_calibration(runs);

The left-hand panel is worth thinking about. Accuracy does change, but not by
much. The filter told that the sonar is wrong 1% of the time guesses
correctly on 35% of steps, against the honest filter's 42%, and both are far
better than the 3% obtained by picking a cell at random. If accuracy were
the only thing you measured, you would conclude that the wrong model was
good enough and ship it.

Looking at the middle panel, we see that the model reports 0.75 confidence while being
correct about a third of the time.


**Question.** A colleague proposes fixing this by setting `epsilon` to 0.5,
on the grounds that it is the most cautious assumption available. Look at
the right-hand end of all three panels. What does this actually do?

*Your answer:*

## Part 7: Looking back

*About 10 minutes.*

In [ ]:
# PROVIDED
if ready(trace, many, runs):
    revealed = reveal(predictions,
                      pd.concat([trace, many], ignore_index=True), runs)
    print(revealed.to_string(index=False))

**Question.** Return to the `why` you wrote in Part 1 and answer it now that
you have the results. Which of the three predictions were you wrong about,
and what in the figures changed your mind?

*Your answer:*

**Question.** In machine learning, which we will learn after Fall break,
models are fitted to data and they are never exactly right. You have just
seen that a model with a single wrong number in it can remain confident while
getting things wrong. In two or three sentences, describe what you would
want to measure about a model before trusting it.

*Your answer:*

In [ ]:
# PROVIDED
if ready(trace, many, runs):
    save_results(pd.concat([trace, many], ignore_index=True), runs)

---

### What you built

Two functions of about six lines each, and a loop around them. That is a
complete Bayes filter. It is also the same algorithm, in the same two steps,
as the Kalman filter that tracks an aircraft and the particle filter in a
self-driving car. Those methods replace the dictionary over thirty cells
with a Gaussian or with a cloud of samples, because a real robot's world
does not have thirty cells in it. The predict step and the update step do
not change.

What you have been tracking is a **hidden variable**. The robot's position
exists and matters, and it is never observed directly. Every reading is a
noisy and partial clue about it. Inferring a hidden quantity from what can
be seen is most of what the rest of this course is about.

`lab5_viterbi.ipynb` is optional and takes about an hour. It asks a question
that this notebook cannot answer: not where the robot is now, but what route
it took. The two questions turn out to have different answers, and the
answer this notebook would give is an impossible one.